In [6]:
import xml.etree.ElementTree as ET
from deep_translator import GoogleTranslator

# Load the XML
tree = ET.parse("CrmTranslations.xml")
root = tree.getroot()

# Define namespaces
ns = {
    'ss': 'urn:schemas-microsoft-com:office:spreadsheet'
}

# Register namespaces for writing
for prefix, uri in ns.items():
    ET.register_namespace(prefix, uri)

# Find the correct worksheet
for worksheet in root.findall('.//ss:Worksheet', ns):
    if worksheet.attrib.get(f'{{{ns["ss"]}}}Name') == "Localized Labels":
        localized_labels_ws = worksheet
        break

# Find all rows in the "Localized Labels" sheet
rows = localized_labels_ws.find('.//ss:Table', ns).findall('ss:Row', ns)

# Identify column index for 1033 and 3082 from header
header_cells = rows[0].findall('ss:Cell', ns)
col_map = {}
for idx, cell in enumerate(header_cells):
    data = cell.find('ss:Data', ns)
    if data is not None and data.text in ["1033", "3082"]:
        col_map[data.text] = idx

# Update rows (skip header)
for row in rows[1:]:
    cells = row.findall('ss:Cell', ns)
    try:
        col_1033 = col_map["1033"]
        col_3082 = col_map["3082"]

        text_1033 = cells[col_1033].find('ss:Data', ns)
        text_3082 = cells[col_3082].find('ss:Data', ns)

        if text_1033 is not None:
            en_text = text_1033.text or ''
            if not text_3082 or not text_3082.text:
                translated = GoogleTranslator(source='en', target='es').translate(en_text)
                if text_3082 is not None:
                    text_3082.text = translated
                else:
                    new_cell = ET.Element(f'{{{ns["ss"]}}}Cell')
                    new_data = ET.Element(f'{{{ns["ss"]}}}Data', {f'{{{ns["ss"]}}}Type': "String"})
                    new_data.text = translated
                    new_cell.append(new_data)
                    row.append(new_cell)
    except Exception as e:
        continue  # Skip problematic rows

# Save the updated XML
tree.write("CrmTranslations_Translated.xml", encoding="utf-8", xml_declaration=True)
